# 03. Étape 1 : Socle Ressources, Chargement Universel & Split Adaptatif
**Projet AutoML – Challenge Machine Learning (M1 IA, Le Mans Université)**

Ce notebook documente et valide la **première étape d’industrialisation** de notre package `automl/`.
Dans le notebook précédent (`02_baseline_automl.ipynb`), nous avions prototypé `detecter_tache` et `load_features_simple`. Nous transférons maintenant cette logique dans deux modules Python robustes :
1. **`automl/config.py`** : lecture sécurisée des limites de ressources (`AUTOML_TIME_BUDGET`, `AUTOML_N_JOBS`, `AUTOML_N_TRIALS`), fixation des graines aléatoires et suivi du budget temps.
2. **`automl/data.py`** : chargement dense/creux, détection automatique des 6 tâches possibles, découpage stratifié adaptatif et garantie du format de prédiction (`formater_predictions`).

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Ajout du dossier racine du projet pour importer directement notre package `automl`
PROJET_ROOT = Path("..").resolve()
if not (PROJET_ROOT / "automl").exists():
    PROJET_ROOT = Path("mon_projet").resolve()
if str(PROJET_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJET_ROOT))

from automl.config import ConfigRessources, ChronoBudget, fixer_graine
from automl.data import (
    resoudre_fichiers_dataset,
    charger_features,
    charger_cible_et_detecter_tache,
    tailles_split_adaptatif,
    decouper_train_val_test,
    formater_predictions,
)

DATASET_DIR = PROJET_ROOT / "dataset"
print(f"[OK] Package automl importé depuis : {PROJET_ROOT}")

[OK] Package automl importé depuis : /Users/zakariafarach/Documents/UNIV/M1_IA/S1/CraftedAutoML/mon_projet


---
## 1. Gestion des ressources et règle Skinner (`automl/config.py`)

Lors de l'évaluation sur Skinner, les contraintes sont transmises via des variables d'environnement. Vérifions comment `ConfigRessources.depuis_env()` réagit — notamment si quelqu'un passe `AUTOML_N_JOBS=-1` (interdit sur le cluster).

In [2]:
fixer_graine(42)

# Simulation d'un environnement Skinner normal
os.environ["AUTOML_TIME_BUDGET"] = "90"
os.environ["AUTOML_N_JOBS"] = "4"
os.environ["AUTOML_N_TRIALS"] = "15"
cfg_normal = ConfigRessources.depuis_env()
print("Config normale :", cfg_normal)

# Simulation d'une valeur dangereuse (-1) -> doit être ramenée à 1
os.environ["AUTOML_N_JOBS"] = "-1"
cfg_safe = ConfigRessources.depuis_env()
print("Sécurité n_jobs=-1 :", cfg_safe)
assert cfg_safe.n_jobs >= 1, "n_jobs=-1 interdit !"

# Nettoyage des variables de test
for k in ("AUTOML_TIME_BUDGET", "AUTOML_N_JOBS", "AUTOML_N_TRIALS"):
    os.environ.pop(k, None)

Config normale : ConfigRessources(time_budget=90.0, n_jobs=4, n_trials=15, seed=42)
Sécurité n_jobs=-1 : ConfigRessources(time_budget=90.0, n_jobs=1, n_trials=15, seed=42)


---
## 2. Chargement universel, détection de tâche & split adaptatif (`automl/data.py`)

Parcourons les 6 jeux de données locaux (`data_A` à `data_F`) pour vérifier que :
- Le format creux `data_F` ($61\,188$ variables) reste en `csr_matrix` sans saturer la mémoire.
- Chaque tâche est détectée sans aucune règle codée en dur.
- Le ratio `val_size / test_size` s'adapte dynamiquement à la taille $N$ du jeu de données.

In [3]:
recap = []
datasets_charges = {}

for nom in ["data_A", "data_B", "data_C", "data_D", "data_E", "data_F"]:
    chemins = resoudre_fichiers_dataset(DATASET_DIR / nom)
    X, fmeta = charger_features(chemins)
    y, tmeta = charger_cible_et_detecter_tache(chemins.solution_path)
    
    val_ratio, test_ratio = tailles_split_adaptatif(X.shape[0])
    split = decouper_train_val_test(X, y, tmeta, seed=42)
    datasets_charges[nom] = (X, y, fmeta, tmeta, split)
    
    recap.append({
        "Dataset": nom,
        "Lignes (N)": X.shape[0],
        "Colonnes (p)": X.shape[1],
        "Format": "Creux (CSR)" if fmeta.is_sparse else "Dense",
        "Tâche détectée": tmeta.task_type,
        "Classes/Sorties": tmeta.n_classes if tmeta.est_classification else tmeta.n_outputs,
        "Ratio Val/Test": f"{int(round(val_ratio*100))}% / {int(round(test_ratio*100))}%",
        "Train / Val / Test": f"{split.y_train.shape[0]} / {split.y_val.shape[0]} / {split.y_test.shape[0]}",
    })

df_recap = pd.DataFrame(recap)
print(df_recap.to_string(index=False))

Dataset  Lignes (N)  Colonnes (p)      Format    Tâche détectée  Classes/Sorties Ratio Val/Test  Train / Val / Test
 data_A       34190            24       Dense        multilabel                3       12% / 8% 27352 / 4102 / 2736
 data_B        5000            16       Dense        regression                1      15% / 10%    3750 / 750 / 500
 data_C       15000          1568       Dense multiclass_onehot               10       12% / 8% 12000 / 1800 / 1200
 data_D        2984           144       Dense            binary                2      15% / 10%    2238 / 447 / 299
 data_E        3140           259       Dense            binary                2      15% / 10%    2355 / 471 / 314
 data_F       13142         61188 Creux (CSR) multiclass_onehot               20       12% / 8% 10513 / 1577 / 1052


---
## 3. Vérification de la stratification (Multi-label & Régression)

Un point clé à expliquer en soutenance : **comment stratifier quand la cible n'est pas une simple classe unique ?**
- Pour le **multi-label** (`data_A`), on encode chaque combinaison de labels (`[1, 1, 0]`, `[0, 1, 1]`, ...) en un identifiant unique afin que les sous-ensembles `Train`, `Val` et `Test` gardent la même fréquence exacte de labels.
- Pour la **régression** (`data_B`), on découpe la cible continue en 10 quantiles afin de conserver la même distribution de prix/valeurs extrêmes.

In [4]:
# 1. Vérification sur data_A (Multi-label : 3 labels binaires)
_, y_A, _, _, split_A = datasets_charges["data_A"]
print("Fréquence moyenne des 3 labels sur data_A :")
print("  - Total :", np.round(y_A.mean(axis=0), 4))
print("  - Train :", np.round(split_A.y_train.mean(axis=0), 4))
print("  - Val   :", np.round(split_A.y_val.mean(axis=0), 4))
print("  - Test  :", np.round(split_A.y_test.mean(axis=0), 4))

# 2. Vérification sur data_B (Régression continue)
_, y_B, _, _, split_B = datasets_charges["data_B"]
print("\nMoyenne et écart-type de la cible sur data_B :")
print(f"  - Train : moyenne={split_B.y_train.mean():.1f}, std={split_B.y_train.std():.1f}")
print(f"  - Val   : moyenne={split_B.y_val.mean():.1f}, std={split_B.y_val.std():.1f}")
print(f"  - Test  : moyenne={split_B.y_test.mean():.1f}, std={split_B.y_test.std():.1f}")

Fréquence moyenne des 3 labels sur data_A :
  - Total : [0.8543 0.6682 0.7608]
  - Train : [0.8543 0.6683 0.7609]
  - Val   : [0.8542 0.6682 0.7608]
  - Test  : [0.8542 0.6681 0.7606]

Moyenne et écart-type de la cible sur data_B :
  - Train : moyenne=207587.9, std=115434.2
  - Val   : moyenne=206577.8, std=113829.6
  - Test  : moyenne=208912.1, std=117288.1


---
## 4. Validation du contrat `predict()` (`formater_predictions`)

Le sujet impose que :
- Si le fichier `.solution` était en **One-Hot** (`data_C`, `data_F`), `predict()` doit impérativement renvoyer une matrice One-Hot (exactement un `1` par ligne).
- Si le fichier `.solution` était une cible 1D (`data_B`, `data_D`), `predict()` doit renvoyer un vecteur 1D avec les bonnes valeurs d'origine.

Vérifions que `formater_predictions()` respecte ce contrat à la lettre.

In [5]:
# Test sur data_C (Multiclasse One-Hot 10 classes)
_, _, _, tmeta_C, split_C = datasets_charges["data_C"]
indices_predits = split_C.y_val[:4]
preds_C = formater_predictions(indices_predits, tmeta_C)
print("data_C (One-Hot) - Forme attendue (4, 10) :", preds_C.shape)
print("Somme par ligne (doit être [1, 1, 1, 1]) :", preds_C.sum(axis=1))
print("Aperçu de la matrice One-Hot générée :\n", preds_C)

# Test sur data_D (Binaire)
_, _, _, tmeta_D, split_D = datasets_charges["data_D"]
preds_D = formater_predictions(split_D.y_val[:6], tmeta_D)
print("\ndata_D (Binaire) - Forme 1D :", preds_D.shape, "| Valeurs :", preds_D)

data_C (One-Hot) - Forme attendue (4, 10) : (4, 10)
Somme par ligne (doit être [1, 1, 1, 1]) : [1 1 1 1]
Aperçu de la matrice One-Hot générée :
 [[0 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0]]

data_D (Binaire) - Forme 1D : (6,) | Valeurs : [0 0 0 1 0 0]


---
## 5. Résumé pour la prochaine étape

- **`config.py` et `data.py` sont opérationnels** et validés sur les 6 datasets réels.
- **Prochaine étape (Étape 2)** : Construire **`automl/preprocessing.py`** pour gérer proprement les valeurs manquantes (`NaN`/`?`), les variables catégorielles (`data_A`), et la vectorisation TF-IDF / normalisation des données creuses (`data_F`).